> ## SOLUTION / ANSWER KEY &mdash; Lab 8.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-assemble-chatbot.ipynb`](../lab-11-assemble-chatbot.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.11 &mdash; Assemble the Customer-Service Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Build billing & tech specialists as real create_agent agents
- Route a real ticket to only the matching specialists, who really answer
- Synthesise one reply, flagged needs_approval for the refund

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-11"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Now assemble the chatbot from your Module 6&ndash;8 pieces (deck slides 15&ndash;17): each specialist is a
**real `create_agent`** with its **own** small tool set. The **supervisor** (`route`) sends the ticket to
only the matching specialists; each **really answers** over its tool; a **synthesiser** weaves their findings
into one reply. Because a refund is **irreversible**, the reply is flagged **`needs_approval`** &mdash;
draft-not-send, now at the *team* level. The routing, synthesis and refund gate are rule-based (they run
offline); the specialists run for real against Groq.

In [ ]:
# The pieces: two @tools (billing/tech), a router, a synthesiser, and a refund gate.
print("assembling: supervisor -> {billing, tech} specialists -> synthesise -> human-gate the refund")

## Build it
Complete `build_specialist` (bind each role's tools), `route` (add the tech specialist), and
`assemble_reply` (flag the refund `needs_approval`). The tools and `synthesize` are written for you.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def route(message):
    m = message.lower()
    engaged = []
    if any(k in m for k in ("charg", "refund", "invoice", "billed")):
        engaged.append("billing")
    if any(k in m for k in ("crash", "bug", "login", "broken")):
        engaged.append("tech")
    return engaged or ["general"]

def refund_intent(message):
    return any(k in message.lower() for k in ("refund", "charged twice", "duplicate", "dispute"))

def synthesize(findings):
    return " ".join(f"[{k}] {findings[k]}" for k in sorted(findings)) if findings else "forwarded to a human agent"

def specialist_reply(agent, message):
    """Invoke a REAL specialist and return its final answer text."""
    result = agent.invoke({"messages": [("user", message)]}, config={"recursion_limit": 8})
    return result["messages"][-1].content

def assemble_reply(findings, involved, message):
    reply = synthesize(findings)
    # a refund is irreversible -> a human approves it (draft-not-send, at the team level)
    needs_human = "billing" in involved and refund_intent(message)
    return {"reply": reply, "status": "needs_approval" if needs_human else "auto_ok", "agents": sorted(findings) or involved}

try:
    # These run offline (no model call): routing + the refund gate are rule-based.
    print("route (two-intent):", route("charged twice for 4471 and the app keeps crashing"))
    print("route (hours)     :", route("what are your hours?"))
    print("refund intent?    :", refund_intent("please refund my invoice"))
    demo = assemble_reply({"billing": "duplicate charge -> refund warranted", "tech": "BUG-231"},
                          ["billing", "tech"], "please refund the duplicate charge")
    print("assembled status  :", demo["status"], "| agents:", demo["agents"])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the real specialists, route a real two-intent ticket, show one specialist's **real trace**, then collect both findings, synthesise, and gate the refund.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        # Build the REAL specialists and route a real two-intent ticket to them.
        billing_agent = build_specialist([lookup_invoice], "billing")
        tech_agent    = build_specialist([known_issues], "tech")
        AGENTS = {"billing": billing_agent, "tech": tech_agent}

        msg = "I was charged twice for order 4471 and the app keeps crashing on login."
        involved = route(msg)
        print("supervisor routed to:", involved)

        # Show ONE real specialist trace, then collect every engaged specialist's finding:
        print("\n-- billing specialist trace --")
        print_trace(billing_agent.invoke({"messages": [("user", msg)]}, config={"recursion_limit": 8}))

        findings = {name: specialist_reply(AGENTS[name], msg) for name in involved if name in AGENTS}
        out = assemble_reply(findings, involved, msg)
        print("\nagents :", out["agents"])
        print("status :", out["status"], "(refund is irreversible -> a human approves)")
        print("reply  :", out["reply"])
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The supervisor routed the two-intent ticket to **both** `billing` and `tech`; each is a real `CompiledStateGraph` that **called its own tool** (see the trace).
- The findings are **synthesised into one reply** tagged by specialist &mdash; one voice, grounded in what each agent actually found.
- Because billing + a refund intent are present, the reply is **`needs_approval`** &mdash; the irreversible action waits for a human. That is the team-level draft-not-send gate.

## Your turn (open task &mdash; no grader)
Send a **pure tech** ticket (*"the app keeps crashing on login"*) through the same flow. **What good looks like:**
only `tech` is engaged, its trace calls `known_issues`, the reply is `auto_ok` (no refund, no human needed).
Then send a refund ticket and confirm it flips to `needs_approval`. (Each live ticket makes a few model calls &mdash;
run a couple, not the whole world, on the free tier.)

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if groq_ready():
    AGENTS = {"billing": build_specialist([lookup_invoice], "billing"),
              "tech":    build_specialist([known_issues], "tech")}
    for msg in ["the app keeps crashing on login",              # pure tech -> auto_ok
                "please refund the duplicate charge on 4471"]:   # refund -> needs_approval
        involved = route(msg)
        findings = {n: specialist_reply(AGENTS[n], msg) for n in involved if n in AGENTS}
        out = assemble_reply(findings, involved, msg)
        print(msg[:36], "->", out["status"], "| agents:", out["agents"])
else:
    print("(add GROQ_API_KEY to .env)")

---
### Key takeaway
A TEAM: a supervisor routes, real specialists (each a create_agent with its own tools) gather, a synthesiser makes one reply -- and the refund waits for a human. Next: run it over a whole suite.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>